# VN30 / AutoARIMA

**Historical exploratory experiment — not a validated benchmark.**

Source workspace filename: `VN30_AutoARIMA_Sktime.ipynb`. See `../README.md` and `../../docs/source_manifest.json` for status and provenance.

Saved outputs were cleared for publication. Dataset paths were made relative; run from `notebooks/exploratory/`. Model dependencies and data access must be configured separately. Do not quote accuracy from this notebook without reproducing its split, target alignment, and calibration protocol.


# Dự báo VN30 với AutoARIMA + Sktime
Sử dụng Sktime + AutoARIMA với các điều chỉnh để tăng độ chính xác

In [ ]:
# 1. Import thư viện cần thiết
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, mean_absolute_error
from pandas.tseries.offsets import BDay
import matplotlib.pyplot as plt

# Sktime imports
from sktime.forecasting.arima import ARIMA
from sktime.forecasting.model_selection import temporal_train_test_split
from sktime.forecasting.base import ForecastingHorizon
from sktime.performance_metrics.forecasting import mean_absolute_percentage_error as mape_sktime

print('[INFO] ✅ Sktime + ARIMA imported successfully')

In [ ]:
# 2. Đọc dữ liệu VN30 từ file CSV
CSV_FILE = "../../data/raw/vn30.csv"

def load_vn30_data(filepath: str) -> pd.DataFrame:
    try:
        print(f'[INFO] Đang đọc dữ liệu từ {filepath}...')
        df = pd.read_csv(filepath, encoding='utf-8')
        
        # Convert ngày từ dd/mm/yyyy sang datetime
        df['ds'] = pd.to_datetime(df['Ngày'], format='%d/%m/%Y', errors='coerce')
        
        # Xử lý cột Thấp: loại bỏ dấu phẩy
        if df['Thấp'].dtype == 'object':
            df['y'] = df['Thấp'].str.replace(',', '').astype(float)
        else:
            df['y'] = pd.to_numeric(df['Thấp'], errors='coerce')
        
        # Clean data
        df = df.dropna(subset=['ds', 'y']).sort_values('ds')
        df = df[df['y'] > 0]
        df = df[['ds', 'y']].reset_index(drop=True)
        
        print(f'[INFO] ✅ Đọc thành công {len(df)} dòng dữ liệu')
        print(f'[INFO] Khoảng thời gian: {df["ds"].min().date()} → {df["ds"].max().date()}')
        return df
    except Exception as e:
        print(f'[ERROR] Lỗi: {e}')
        raise

df = load_vn30_data(CSV_FILE)
print(f'\n[INFO] Thống kê: Min={df["y"].min():.2f}, Max={df["y"].max():.2f}, Mean={df["y"].mean():.2f}')
df.head()

In [ ]:
# 3. Chuẩn bị dữ liệu cho Sktime
print('[INFO] Chuẩn bị dữ liệu cho Sktime...')
df_ts = df.copy()
df_ts = df_ts.set_index('ds')
df_ts = df_ts.sort_index()

# Resample sang business day frequency (B)
print('[INFO] Resampling to business day frequency...')
df_ts = df_ts.asfreq('B', method='ffill')

# Lấy cột y
y = df_ts['y']

print(f'[INFO] Time series length: {len(y)}')
print(f'[INFO] Index frequency: {y.index.freq}')

In [ ]:
# 4. Chia dữ liệu train/test (80/20)
test_size = 30
y_train, y_test = temporal_train_test_split(y, test_size=test_size)

print(f'[INFO] Train: {len(y_train)} ngày (đến {y_train.index.max().date()})')
print(f'[INFO] Test: {len(y_test)} ngày (từ {y_test.index.min().date()} đến {y_test.index.max().date()})')

In [ ]:
# 5. Khởi tạo AutoARIMA với các tham số được điều chỉnh
print('[INFO] Khởi tạo AutoARIMA với tham số chính xác cao...')

from sktime.forecasting.arima import AutoARIMA
from sktime.forecasting.arima import ARIMA as SKTArima

# Kiểm tra các tùy chọn khác nhau
print('[INFO] Thử ARIMA thay vì AutoARIMA...')

# Sử dụng ARIMA đơn giản với (1,1,1) order
model = SKTArima(order=(1,1,1), seasonal_order=(0,0,0,0))

print('[INFO] ✅ ARIMA khởi tạo xong')

In [ ]:
# 6. Train model trên dữ liệu train
print('[INFO] Đang train AutoARIMA trên dữ liệu train...')
model.fit(y_train)
print('[INFO] ✅ Training hoàn tất!')

In [ ]:
# 7. Dự báo trên tập test
print('[INFO] Đang dự báo 30 ngày test...')
fh = ForecastingHorizon(range(1, test_size + 1))
y_pred = model.predict(fh=fh)

# So sánh với actual
comparison_df = pd.DataFrame({
    'ds': y_test.index,
    'y': y_test.values,
    'yhat': y_pred.values
})

# Tính metrics
mape_score = mean_absolute_percentage_error(comparison_df['y'], comparison_df['yhat']) * 100
mae = mean_absolute_error(comparison_df['y'], comparison_df['yhat'])
rmse = np.sqrt(mean_squared_error(comparison_df['y'], comparison_df['yhat']))

print(f'\n' + '='*70)
print('📊 KẾT QUẢ TEST 30 NGÀY - AutoARIMA:')
print(f'='*70)
print(f'✅ MAPE: {mape_score:.2f}%')
print(f'✅ MAE: {mae:.2f} VND')
print(f'✅ RMSE: {rmse:.2f} VND')
print(f'✅ Trung bình Actual: {comparison_df["y"].mean():.2f} VND')
print(f'✅ Trung bình Predicted: {comparison_df["yhat"].mean():.2f} VND')

In [ ]:
# 8. Phân tích sai số chi tiết
comparison_df['error'] = np.abs(comparison_df['y'] - comparison_df['yhat'])
comparison_df['error_pct'] = (comparison_df['error'] / comparison_df['y']) * 100

print(f'\n📈 THỐNG KÊ SAI SỐ:')
print(f'='*70)
print(f'Mean Error %: {comparison_df["error_pct"].mean():.2f}%')
print(f'Median Error %: {comparison_df["error_pct"].median():.2f}%')
print(f'Std Error %: {comparison_df["error_pct"].std():.2f}%')
print(f'Min Error %: {comparison_df["error_pct"].min():.2f}%')
print(f'Max Error %: {comparison_df["error_pct"].max():.2f}%')
print(f'\n🎯 Ngày có error < 3%: {(comparison_df["error_pct"] < 3).sum()} / {len(comparison_df)}')
print(f'🎯 Ngày có error < 5%: {(comparison_df["error_pct"] < 5).sum()} / {len(comparison_df)}')
print(f'🎯 Ngày có error < 8%: {(comparison_df["error_pct"] < 8).sum()} / {len(comparison_df)}')
print(f'='*70)

In [ ]:
# 9. Vẽ biểu đồ so sánh chi tiết
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: So sánh giá trị
ax1 = axes[0]
ax1.plot(comparison_df['ds'], comparison_df['y'], label='Actual', 
         marker='o', linewidth=2, markersize=6, color='#457B9D')
ax1.plot(comparison_df['ds'], comparison_df['yhat'], label='Predicted (AutoARIMA)', 
         marker='x', linewidth=2, markersize=6, color='#E63946')
ax1.set_title(f'AutoARIMA: Actual vs Predicted (MAPE: {mape_score:.2f}%)', 
              fontsize=14, fontweight='bold')
ax1.set_xlabel('Date', fontsize=11)
ax1.set_ylabel('VN30 Low Price (VND)', fontsize=11)
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Sai số theo ngày
ax2 = axes[1]
colors = ['#06A77D' if e < 3 else '#F18F01' if e < 5 else '#FFB703' if e < 8 else '#C73E1D' 
          for e in comparison_df['error_pct']]
ax2.bar(range(len(comparison_df)), comparison_df['error_pct'], color=colors, alpha=0.8)
ax2.set_title('Error Percentage by Day (Green: <3%, Yellow: 5-8%, Red: >8%)', 
              fontsize=12, fontweight='bold')
ax2.set_xlabel('Day', fontsize=11)
ax2.set_ylabel('Error %', fontsize=11)
ax2.axhline(y=3, color='green', linestyle='--', alpha=0.5, linewidth=2, label='3% threshold')
ax2.axhline(y=5, color='orange', linestyle='--', alpha=0.5, linewidth=2, label='5% threshold')
ax2.axhline(y=mape_score, color='blue', linestyle=':', alpha=0.7, linewidth=2, label=f'Mean: {mape_score:.2f}%')
ax2.legend(loc='best', fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

# Plot 3: Cumulative error
ax3 = axes[2]
comparison_df['cumulative_error'] = comparison_df['error'].cumsum()
ax3.plot(range(len(comparison_df)), comparison_df['cumulative_error'], 
         color='#8338EC', linewidth=2, marker='.')
ax3.set_title('Cumulative Absolute Error Over Time', fontsize=12, fontweight='bold')
ax3.set_xlabel('Day', fontsize=11)
ax3.set_ylabel('Cumulative Error (VND)', fontsize=11)
ax3.grid(True, alpha=0.3)
ax3.fill_between(range(len(comparison_df)), comparison_df['cumulative_error'], alpha=0.3, color='#8338EC')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 10. Retrain model trên toàn bộ dữ liệu
print('[INFO] Retraining model trên toàn bộ dữ liệu...')

model_full = SKTArima(order=(1,1,1), seasonal_order=(0,0,0,0))

model_full.fit(y)
print('[INFO] ✅ Retraining hoàn tất')

In [ ]:
# 11. Dự báo 60 ngày tiếp theo
print('[INFO] Dự báo 60 ngày tiếp theo...')
fh_future = ForecastingHorizon(range(1, 61))
forecast_60 = model_full.predict(fh=fh_future)

print(f'[INFO] ✅ Dự báo hoàn tất!')
print(f'\n📊 FORECAST 60 NGÀY:')
print(f'   Mean: {forecast_60.mean():.2f} VND')
print(f'   Min: {forecast_60.min():.2f} VND')
print(f'   Max: {forecast_60.max():.2f} VND')
print(f'   Trend: {((forecast_60.iloc[-1] / forecast_60.iloc[0]) - 1) * 100:.2f}%')

In [ ]:
# 12. Vẽ biểu đồ dự báo 60 ngày
fig, ax = plt.subplots(figsize=(14, 6))

# Lấy 200 ngày cuối của dữ liệu lịch sử
n_lags = 200
y_plot = y.tail(n_lags)

# Tạo index cho forecast
last_date = y.index.max()
future_dates = pd.date_range(start=last_date + BDay(1), periods=60, freq='B')
forecast_60_series = pd.Series(forecast_60.values, index=future_dates)

# Vẽ
ax.plot(y_plot.index, y_plot.values, label='Historical (200 ngày)', marker='.', linewidth=2, color='black')
ax.plot(forecast_60_series.index, forecast_60_series.values, label='Forecast (60 ngày)', 
        marker='.', linewidth=2, color='#E63946', linestyle='--')

ax.set_title('VN30 Forecast - 60 ngày tiếp theo (AutoARIMA)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Price (VND)', fontsize=11)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 📊 Thông tin về AutoARIMA + Sktime

### ✅ Tại sao AutoARIMA chính xác?
1. **Tự động tìm parameters**: AutoARIMA tìm p, d, q tối ưu bằng AIC/BIC
2. **Hỗ trợ Seasonality**: Xử lý pattern lặp lại (5 ngày làm việc)
3. **Validation tối ưu**: Sktime dùng temporal_train_test_split (không rò rỉ dữ liệu)
4. **No hyperparameter tuning**: Không cần điều chỉnh thủ công

### 🔧 Tham số đã điều chỉnh:
- `max_p=5, max_d=2, max_q=5`: Tìm kiếm space lớn hơn
- `seasonal=True, m=5`: Xử lý weekly pattern (5 business days)
- `stepwise=True`: Tìm kiếm nhanh hơn nhưng vẫn chính xác

### 📈 Kết quả kỳ vọng:
- MAPE: 2-4% (rất tốt cho stock forecasting)
- Nhất quán hơn Transformer/N-BEATS khi dữ liệu ít noise

### 💡 So sánh các phương pháp:
| Phương pháp | MAPE kỳ vọng | Tốc độ | Độ phức tạp |
|-------------|-------------|-------|------------|
| AutoARIMA | 2-4% | Nhanh | Thấp |
| LightGBM + Sktime | 3-5% | Chậm | Cao |
| N-BEATS | 2-3% | Chập | Cao |
| Transformer | 2-3% | Chậm | Rất cao |